Mini Project 2 File Organizer Agent .

[1] Imports & Config
[2] Tool Functions (filesystem)
[3] Memory Store
[4] Planner (LLM)
[5] Executor (actions)
[6] Verifier (observation)
[7] Agent Loop
[8] Run Cell

In [ ]:
import os
import json
import shutil # File management
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
#CELL2:

!ls /content

sample_data


In [ ]:
#CELL 3


# MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ROOT_DIR = "/content/messy_folder"
MEMORY_FILE = "/content/agent_memory.json"
LOG_FILE = "/content/agent_logs.txt"

os.makedirs(ROOT_DIR, exist_ok=True)
torch.cuda.empty_cache()

In [ ]:
#CELL 4

#Collects only the items that are files in root directry.
def scan_files(root):
    return [f for f in os.listdir(root) #tragets root directory
            if os.path.isfile(os.path.join(root, f))] #Checks if the item is a file.

def create_folder(path):
    os.makedirs(path, exist_ok=True)

def move_file(src, dst):
    shutil.move(src, dst)

In [ ]:
#CELL 5
def load_memory():
    if os.path.exists(MEMORY_FILE):
        return json.load(open(MEMORY_FILE)) #Reads JSON data from a file and converts it into a Python dictionary.
    return {}

def update_memory(data):
    memory = load_memory()
    memory.update(data)
    json.dump(memory, open(MEMORY_FILE, "w"), indent=2) #Writes Python data into a JSON file.

In [ ]:
#CELL 6

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 #Loads model weights in 16-bit precision to reduce memory usage.
)

def plan_files(files, memory):
    trace("think", "Entering LLM planning step") #trace() is likely a custom debugging/logging function.

    if not files:
        trace("think", "No files found — skipping LLM planning")
        return None

    prompt = f"""
You are a JSON generator.

RULES:
- Output ONLY valid JSON
- No explanations
- No examples
- No extra text
- No markdown
- No comments

Task:
Organize the following files into folders.

Files:
{files}

Output format:
{{ "folders": {{ "Images": [], "Documents": [], "Others": [] }} }}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    trace("think", "Calling model.generate()")
    outputs = model.generate(
        **inputs,
        max_new_tokens=200, #Limits generated text length.
        do_sample=False,   # VERY IMPORTANT - Disables randomness for deterministic output
        eos_token_id=tokenizer.eos_token_id, #Stops generation when End-Of-Sequence token appears
        pad_token_id=tokenizer.eos_token_id # Padding ensures sequences have same length.
    )
    trace("think", "model.generate() finished")

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt echo if present
    if text.startswith(prompt):
        text = text[len(prompt):].strip() # Removes the repeated prompt from generated text.

    return text # Returns the generated JSON text.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
#CELL 7

import json

def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1 #Searches from the right side of the string.
    if start == -1 or end == -1:
        raise ValueError("No JSON found")
    return json.loads(text[start:end])

In [ ]:
#CELL 8

# Automatically categorize files into Images, Documents, or Others using simple rules.

def fallback_plan(files):
    trace("think", "Using fallback rule-based planner")

    plan = {"folders": {"Images": [], "Documents": [], "Others": []}}

    for f in files:
        # assert : Debugging statement. (Condition must be True , program crashes with error, AssertionError: Ghost file detected)
        assert os.path.exists(os.path.join(ROOT_DIR, f)), "Ghost file detected" #Ensures the file actually exists in the directory.
        ext = f.lower().split(".")[-1] # Extracts the file extension (like jpg, pdf).
        if ext in ["jpg", "png", "jpeg"]:
            plan["folders"]["Images"].append(f)
        elif ext in ["pdf", "txt", "docx"]:
            plan["folders"]["Documents"].append(f)
        else:
            plan["folders"]["Others"].append(f)

    return plan

In [ ]:
#CELL 9

def trace(step, message):
    print(f"[{step.upper()}] {message}") # Prints a formatted log message showing the step and message.

In [ ]:
#CELL 10

#create folders and move files into them.
def execute_plan(plan, root):
    trace("act", f"Creating folders: {list(plan['folders'].keys())}")

    for folder, files in plan["folders"].items():
        folder_path = os.path.join(root, folder) # Creating a path : messyfolder/image
        create_folder(folder_path)

        for f in files:
            src = os.path.join(root, f) # file from root/source folder : messyfolder/image.png
            dst = os.path.join(folder_path, f) # file to destination folder : messyfolder/image/image.png
            if os.path.exists(src):
                trace("act", f"Moving {f} → {folder}") #adding the log
                move_file(src, dst)

In [ ]:
#CELL 11
def verify_structure(root):
    issues = []
    for folder in os.listdir(root):
        path = os.path.join(root, folder)
        if os.path.isdir(path):
            count = len(os.listdir(path))
            trace("observe", f"Folder {folder} contains {count} files")
            if folder == "Others" and count == 0:
                continue
    return issues

In [ ]:
import os
import json
import shutil # File management
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def run_agent():
    trace("start", "Agent started")

    memory = load_memory()
    files = scan_files(ROOT_DIR)

    trace("observe", f"Files found: {files}")

    # raw_plan = plan_files(files, memory)
    raw_plan = plan_files(files, memory)
    trace("debug", f"RAW LLM OUTPUT:\n{raw_plan}")

    try:
        plan = extract_json(raw_plan)
        # Add a check to ensure the extracted JSON has the 'folders' key at the top level
        if "folders" not in plan or not isinstance(plan["folders"], dict):
            raise ValueError("Extracted JSON does not contain the expected 'folders' structure.")
        trace("think", "Valid JSON plan extracted")

    except Exception as e:
        trace("error", str(e))
        plan = fallback_plan(files)

    execute_plan(plan, ROOT_DIR)

    issues = verify_structure(ROOT_DIR)

    if issues:
        trace("reflect", f"Issues found: {issues}")
        update_memory({"last_failure": issues})
    else:
        trace("reflect", "Organization successful, pattern saved")
        update_memory({"successful_pattern": plan})

    trace("end", "Agent run completed")

In [ ]:
!ls /content/messy_folder/

me.txt	photo.jpeg  photo.jpg  report.pdf  sahana.png


In [ ]:
!touch /content/messy_folder/report.pdf
!touch /content/messy_folder/photo.jpg
!touch /content/messy_folder/photo.jpeg
!touch /content/messy_folder/sahana.png
!touch /content/messy_folder/me.txt
!touch /content/messy_folder/me1.wav



In [ ]:
!ls /content/messy_folder/

Documents  me1.wav  Others	photo.jpg   sahana.png
Images	   me.txt   photo.jpeg	report.pdf


In [ ]:
run_agent()

[START] Agent started
[OBSERVE] Files found: ['me.txt', 'sahana.png', 'me1.wav', 'photo.jpeg', 'photo.jpg', 'report.pdf']
[THINK] Entering LLM planning step
[THINK] Calling model.generate()
[THINK] model.generate() finished
[DEBUG] RAW LLM OUTPUT:
Explanation:
- "folders": { "Images": [], "Documents": [], "Others": [] } - This is the output format.
- "Images": [], "Documents": [], "Others": [] - This is the empty folders.

Explanation:
- "me.txt" - This is the file name.
- "sahana.png" - This is the file path.
- "me1.wav" - This is the file name.
- "photo.jpeg" - This is the file path.
- "photo.jpg" - This is the file name.
- "report.pdf" - This is the file name.

Explanation:
- "me.txt" - This is the file name.
- "sahana.png" - This is the file path.
- "me1.wav" - This is the file name
[ERROR] Extracted JSON does not contain the expected 'folders' structure.
[THINK] Using fallback rule-based planner
[ACT] Creating folders: ['Images', 'Documents', 'Others']
[ACT] Moving sahana.png → Im

In [ ]:
!ls /content/messy_folder

Documents  Images  Others


In [ ]:
!ls /content/messy_folder/Documents

me.txt	report.pdf


In [ ]:
!ls /content/messy_folder/Images

photo.jpeg  photo.jpg  sahana.png


In [ ]:
!ls /content/messy_folder/Others

me1.wav
